# DDPO Baseline — Results Analysis

This notebook loads the metrics history from a DDPO training run and produces:
1. **PickScore curves** — training reward over time
2. **Diversity curves** — LPIPS diversity per category over time
3. **Reward vs. diversity scatter** — the collapse trade-off
4. **Qualitative grids** — before/after image comparison

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from IPython.display import Image, display

sns.set_theme(style="whitegrid", font_scale=1.2)
%matplotlib inline

# Configure paths. Colab training saves to Drive; local training saves to ../outputs/baseline.
CANDIDATE_OUTPUT_DIRS = [
    "/content/drive/MyDrive/ddpo_baseline/run1",
    "../outputs/baseline",
]

def find_metrics_path(output_dirs):
    for output_dir in output_dirs:
        metrics_path = os.path.join(output_dir, "metrics_history.json")
        if os.path.exists(metrics_path):
            return output_dir, metrics_path

        ckpt_dir = os.path.join(output_dir, "checkpoints")
        if os.path.isdir(ckpt_dir):
            step_dirs = sorted(
                d for d in os.listdir(ckpt_dir)
                if d.startswith("step_") and os.path.isdir(os.path.join(ckpt_dir, d))
            )
            for step_dir in reversed(step_dirs):
                metrics_path = os.path.join(ckpt_dir, step_dir, "metrics_history.json")
                if os.path.exists(metrics_path):
                    return output_dir, metrics_path

    searched = "\n".join(output_dirs)
    raise FileNotFoundError(f"Could not find metrics_history.json. Searched:\n{searched}")

OUTPUT_DIR, METRICS_PATH = find_metrics_path(CANDIDATE_OUTPUT_DIRS)
print(f"Using output dir: {OUTPUT_DIR}")
print(f"Using metrics:    {METRICS_PATH}")

with open(METRICS_PATH) as f:
    metrics_history = json.load(f)

print(f"Loaded {len(metrics_history)} evaluation checkpoints")
metrics_history[0]

In [ ]:
# ── Extract time series ──────────────────────────────────────
steps = [m["step"] for m in metrics_history]

categories = ["constrained", "common_subject", "open_ended"]
colors = {"constrained": "#2196F3", "common_subject": "#FF9800", "open_ended": "#E91E63"}

pick_overall = [m["eval/overall/pickscore"] for m in metrics_history]
div_overall = [m["eval/overall/diversity"] for m in metrics_history]

pick_by_cat = {}
div_by_cat = {}
for cat in categories:
    pick_by_cat[cat] = [m[f"eval/{cat}/pickscore"] for m in metrics_history]
    div_by_cat[cat] = [m[f"eval/{cat}/diversity"] for m in metrics_history]

In [ ]:
# ── 1. PickScore over training ──────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(steps, pick_overall, "k-o", linewidth=2, markersize=5, label="Overall")
for cat in categories:
    ax.plot(steps, pick_by_cat[cat], "-s", color=colors[cat],
            markersize=4, alpha=0.8, label=cat.replace("_", " ").title())

ax.set_xlabel("Training Step")
ax.set_ylabel("Mean PickScore")
ax.set_title("PickScore Over Training (Higher = Better)")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "pickscore_curve.png"), dpi=150)
plt.show()

In [ ]:
# ── 2. Diversity (LPIPS) over training ──────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(steps, div_overall, "k-o", linewidth=2, markersize=5, label="Overall")
for cat in categories:
    ax.plot(steps, div_by_cat[cat], "-s", color=colors[cat],
            markersize=4, alpha=0.8, label=cat.replace("_", " ").title())

ax.set_xlabel("Training Step")
ax.set_ylabel("Mean Pairwise LPIPS")
ax.set_title("Intra-Prompt Diversity Over Training (Higher = More Diverse)")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "diversity_curve.png"), dpi=150)
plt.show()

In [ ]:
# ── 3. Diversity drop table ─────────────────────────────────
print("\n=== Diversity Collapse Summary ===")
print(f"{'Category':<20} {'Start':>10} {'End':>10} {'Change':>10} {'% Drop':>10}")
print("-" * 62)

for cat in categories:
    start = div_by_cat[cat][0]
    end = div_by_cat[cat][-1]
    change = end - start
    pct = (change / start) * 100 if start != 0 else 0
    print(f"{cat:<20} {start:>10.4f} {end:>10.4f} {change:>+10.4f} {pct:>+9.1f}%")

start_o = div_overall[0]
end_o = div_overall[-1]
change_o = end_o - start_o
pct_o = (change_o / start_o) * 100 if start_o != 0 else 0
print(f"{'overall':<20} {start_o:>10.4f} {end_o:>10.4f} {change_o:>+10.4f} {pct_o:>+9.1f}%")

In [ ]:
# ── 4. Reward vs Diversity scatter ──────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for cat in categories:
    ax.scatter(pick_by_cat[cat], div_by_cat[cat], c=colors[cat],
               label=cat.replace("_", " ").title(), s=60, alpha=0.8, edgecolors="white")
    # Arrow from start to end
    ax.annotate("", xy=(pick_by_cat[cat][-1], div_by_cat[cat][-1]),
                xytext=(pick_by_cat[cat][0], div_by_cat[cat][0]),
                arrowprops=dict(arrowstyle="->", color=colors[cat], lw=1.5))

ax.set_xlabel("PickScore")
ax.set_ylabel("LPIPS Diversity")
ax.set_title("Reward vs. Diversity Trade-off\n(arrows show training progression)")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "reward_vs_diversity.png"), dpi=150)
plt.show()

In [ ]:
# ── 5. Before/after grid ────────────────────────────────────
grid_path = os.path.join(OUTPUT_DIR, "before_after_grid.png")
if os.path.exists(grid_path):
    print("Before/After Comparison Grid:")
    display(Image(filename=grid_path))
else:
    print(f"Grid not found at {grid_path}. Run training first.")

In [ ]:
# ── 6. Per-step eval image grids ────────────────────────────
eval_dir = os.path.join(OUTPUT_DIR, "eval_images")
if os.path.exists(eval_dir):
    step_dirs = sorted([d for d in os.listdir(eval_dir) if d.startswith("step_")])
    print(f"Found eval images for {len(step_dirs)} steps: {step_dirs}")
    
    # Show first and last step grids side by side
    for sd in [step_dirs[0], step_dirs[-1]]:
        imgs = sorted(os.listdir(os.path.join(eval_dir, sd)))
        if imgs:
            print(f"\n--- {sd} ({len(imgs)} images) ---")
            for img in imgs[:3]:  # show first 3
                display(Image(filename=os.path.join(eval_dir, sd, img), width=600))
else:
    print("No eval images found. Run training first.")